## Tool 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.04.02 </div>
<div style="text-align: right"> last update : 2026.04.02 </div>

개정 이력  
- `2026.04.02` : 노트북 최초 작성 — ReAct Orchestrator + Worker + Tools 아키텍처 반영

### 개요

현재 에이전트 아키텍처는 **14개 Tool**을 4개 카테고리로 분류합니다:

| 카테고리 | Tool | 인프라 요구 |
|----------|------|------------|
| CALCULATION_TOOLS | `calculate_roi`, `calculate_weighted_score`, `estimate_timeline`, `assess_risk_matrix` | 없음 (순수 함수) |
| UTILITY_TOOLS | `format_currency`, `current_date` | 없음 |
| DATA_TOOLS | `get_team_members`, `get_scoring_criteria`, `get_company_settings`, `search_company_context`, `search_similar_projects`, `get_past_deal_analysis` | DB + Pinecone |
| EXTERNAL_TOOLS | `web_search`, `fetch_notion_deal` | 외부 API key |

이 노트북은 **인프라 불필요 Tool → DB 의존 Tool → 외부 API Tool** 순서로 테스트합니다.

Worker들은 이 Tool들을 ReAct loop에서 자율적으로 호출합니다.

In [ ]:
import json
import sys

from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, "../../")

In [ ]:
# Tool 카테고리 확인
from backend.app.agent.tools import (
    ALL_TOOLS,
    CALCULATION_TOOLS,
    DATA_TOOLS,
    EXTERNAL_TOOLS,
    UTILITY_TOOLS,
)

print(f"CALCULATION_TOOLS ({len(CALCULATION_TOOLS)}): {[t.name for t in CALCULATION_TOOLS]}")
print(f"UTILITY_TOOLS ({len(UTILITY_TOOLS)}): {[t.name for t in UTILITY_TOOLS]}")
print(f"DATA_TOOLS ({len(DATA_TOOLS)}): {[t.name for t in DATA_TOOLS]}")
print(f"EXTERNAL_TOOLS ({len(EXTERNAL_TOOLS)}): {[t.name for t in EXTERNAL_TOOLS]}")
print(f"ALL_TOOLS ({len(ALL_TOOLS)}): {[t.name for t in ALL_TOOLS]}")

---
### 1. CALCULATION_TOOLS (인프라 불필요)

순수 결정적(deterministic) 함수들로, LLM이 산술 계산을 직접 하지 않도록 보장합니다.

#### 1-1. calculate_roi — 수익률/마진 계산

In [ ]:
from backend.app.agent.tools import calculate_roi

# Case 1: 정상 수익 (마진 30%+)
result = await calculate_roi.ainvoke({
    "contract_amount": 20000,  # 2억원
    "estimated_cost": 13000,   # 1.3억원
    "duration_months": 5.0,
})
roi = json.loads(result)
print("=== Case 1: 고마진 프로젝트 ===")
print(json.dumps(roi, ensure_ascii=False, indent=2))
print(f"마진율: {roi['margin_pct']}%, 등급: {roi['profitability_rating']}")

In [ ]:
# Case 2: 손실 프로젝트 (마진 음수)
result = await calculate_roi.ainvoke({
    "contract_amount": 8000,
    "estimated_cost": 10000,
    "duration_months": 6.0,
})
roi_loss = json.loads(result)
print(f"마진율: {roi_loss['margin_pct']}%, 등급: {roi_loss['profitability_rating']}")
assert roi_loss['profitability_rating'] == 'loss', "음수 마진 → loss 등급이어야 함"

#### 1-2. calculate_weighted_score — 가중 점수 재계산

In [ ]:
from backend.app.agent.tools import calculate_weighted_score

# 7개 기준 점수 입력
sample_scores = [
    {"criterion": "기술 적합성", "score": 80, "weight": 0.20},
    {"criterion": "수익성", "score": 65, "weight": 0.20},
    {"criterion": "리소스 가용성", "score": 70, "weight": 0.15},
    {"criterion": "납기 리스크", "score": 60, "weight": 0.10},
    {"criterion": "요구사항 명확성", "score": 75, "weight": 0.10},
    {"criterion": "전략적 가치", "score": 85, "weight": 0.15},
    {"criterion": "고객 리스크", "score": 70, "weight": 0.10},
]

result = await calculate_weighted_score.ainvoke({"scores": sample_scores})
ws = json.loads(result)
print(json.dumps(ws, ensure_ascii=False, indent=2))
print(f"\n총점: {ws['total_score']}, 판정: {ws['verdict']}, 가중치 합: {ws['weight_sum']}")

In [ ]:
# Verdict 임계값 검증
for score_val, expected_verdict in [(90, "go"), (70, "go"), (55, "conditional_go"), (40, "conditional_go"), (30, "no_go")]:
    test_scores = [{"criterion": "test", "score": score_val, "weight": 1.0}]
    result = await calculate_weighted_score.ainvoke({"scores": test_scores})
    v = json.loads(result)["verdict"]
    status = "✓" if v == expected_verdict else "✗"
    print(f"  {status} score={score_val} → {v} (expected: {expected_verdict})")

#### 1-3. estimate_timeline — PERT 기반 일정 추정

In [ ]:
from backend.app.agent.tools import estimate_timeline

# Case 1: 복잡도 높음, 대규모 팀, 외부 의존성 있음
result = await estimate_timeline.ainvoke({
    "base_duration_months": 5.0,
    "team_size": 7,
    "complexity": "high",
    "has_external_dependencies": True,
})
tl = json.loads(result)
print("=== High complexity, large team, external deps ===")
print(json.dumps(tl, ensure_ascii=False, indent=2))
print(f"\nPERT 추정: {tl['pert_estimate_months']}개월 + 버퍼 {tl['buffer_months']}개월 = 권장 {tl['recommended_months']}개월")

In [ ]:
# Case 2: 단순 프로젝트
result = await estimate_timeline.ainvoke({
    "base_duration_months": 3.0,
    "team_size": 3,
    "complexity": "low",
    "has_external_dependencies": False,
})
tl_simple = json.loads(result)
print(f"단순 프로젝트: base=3개월 → 권장 {tl_simple['recommended_months']}개월")
assert tl_simple['recommended_months'] < tl['recommended_months'], "단순 프로젝트가 더 짧아야 함"

#### 1-4. assess_risk_matrix — 확률 × 영향도 매트릭스

In [ ]:
from backend.app.agent.tools import assess_risk_matrix

sample_risks = [
    {"category": "기술", "item": "신규 기술 도입 리스크", "probability": "HIGH", "impact": "HIGH"},
    {"category": "일정", "item": "일정 초과 리스크", "probability": "MEDIUM", "impact": "HIGH"},
    {"category": "재무", "item": "비용 초과 리스크", "probability": "LOW", "impact": "MEDIUM"},
    {"category": "고객", "item": "요구사항 변경", "probability": "MEDIUM", "impact": "MEDIUM"},
    {"category": "범위", "item": "범위 확장", "probability": "HIGH", "impact": "LOW"},
]

result = await assess_risk_matrix.ainvoke({"risks": sample_risks})
rm = json.loads(result)
print(json.dumps(rm, ensure_ascii=False, indent=2))

# level 검증: HIGH×HIGH → CRITICAL
first_risk = rm['risks'][0]
assert first_risk['level'] == 'CRITICAL', f"HIGH×HIGH는 CRITICAL이어야 함, got: {first_risk['level']}"
print(f"\n✓ HIGH×HIGH → {first_risk['level']}")
print(f"전체 리스크 프로파일: {rm['summary']['overall_risk_profile']}")

---
### 2. UTILITY_TOOLS (인프라 불필요)

In [ ]:
from backend.app.agent.tools import current_date, format_currency

# format_currency 테스트
test_amounts = [0, 500, 8000, 10000, 15000, 50000, -3000]
for amount in test_amounts:
    result = await format_currency.ainvoke({"amount": amount})
    print(f"  {amount:>8}만원 → {result}")

# current_date 테스트
date_result = await current_date.ainvoke({})
print(f"\n현재 날짜: {date_result}")

---
### 3. DATA_TOOLS (DB + Pinecone 필요)

**사전 요구사항:** `make docker-up && make migrate && make seed` + Pinecone 설정

DATA_TOOLS는 `ToolContext`를 통해 DB session과 vector store에 접근합니다.

In [ ]:
# ToolContext 초기화 — DATA_TOOLS 사용 전 필수
from backend.app.agent.tools._context import init_tool_context
from backend.app.db.session import AsyncSessionLocal
from backend.app.db.vector_store import CompanyContextStore, ProjectHistoryStore

init_tool_context(
    session_factory=AsyncSessionLocal,
    company_context_store=CompanyContextStore(),
    project_history_store=ProjectHistoryStore(),
)
print("✓ ToolContext 초기화 완료")

#### 3-1. get_team_members — 팀원 목록

In [ ]:
from backend.app.agent.tools import get_team_members

result = await get_team_members.ainvoke({})
members = json.loads(result)
print(f"팀원 수: {len(members)}")
for m in members:
    avail = "가용" if m['is_available'] else f"불가 (from: {m.get('available_from', 'N/A')})"
    print(f"  {m['name']} ({m['role']}) — 월 {m['monthly_rate']}만원, {avail}")

#### 3-2. get_scoring_criteria — 평가 기준

In [ ]:
from backend.app.agent.tools import get_scoring_criteria

result = await get_scoring_criteria.ainvoke({})
criteria = json.loads(result)
print(f"평가 기준 수: {len(criteria)}")
total_weight = 0
for c in criteria:
    total_weight += c['weight']
    print(f"  {c['name']}: weight={c['weight']}, {c['description'][:50]}...")
print(f"가중치 합계: {total_weight}")

#### 3-3. get_company_settings — 회사 설정

In [ ]:
from backend.app.agent.tools import get_company_settings

result = await get_company_settings.ainvoke({})
settings = json.loads(result)
for key, value in settings.items():
    preview = value[:100] + "..." if len(value) > 100 else value
    print(f"  [{key}] {preview}")

#### 3-4. search_company_context — 벡터 검색 (회사 컨텍스트)

In [ ]:
from backend.app.agent.tools import search_company_context

result = await search_company_context.ainvoke({
    "query": "Vision AI 기반 제조업 프로젝트 기술 스택",
    "top_k": 3,
})
ctx_result = json.loads(result)
print(f"검색 결과 문서 수: {len(ctx_result.get('documents', []))}")
print(f"\n포맷된 컨텍스트:\n{ctx_result.get('formatted', 'N/A')[:500]}")

#### 3-5. search_similar_projects — 벡터 검색 (유사 프로젝트)

In [ ]:
from backend.app.agent.tools import search_similar_projects

result = await search_similar_projects.ainvoke({
    "query": "제조업 공장 설비 예지보전 AI 시계열 예측 모델",
    "top_k": 3,
    "industry": "제조업",
})
projects = json.loads(result)
print(f"유사 프로젝트 수: {len(projects) if isinstance(projects, list) else 'error'}")
if isinstance(projects, list):
    for p in projects:
        print(f"  {p.get('project_name', 'N/A')} — similarity: {p.get('similarity_score', 'N/A')}")

#### 3-6. get_past_deal_analysis — 과거 분석 결과 조회

In [ ]:
from backend.app.agent.tools import get_past_deal_analysis

# 존재하지 않는 deal_id로 에러 케이스 테스트
import uuid

result = await get_past_deal_analysis.ainvoke({"deal_id": str(uuid.uuid4())})
past = json.loads(result)
print(f"존재하지 않는 deal 조회: {past}")
assert "error" in past, "존재하지 않는 deal은 에러를 반환해야 함"

---
### 4. EXTERNAL_TOOLS (외부 API 필요)

**사전 요구사항:** Tavily API key, Notion integration 설정

#### 4-1. web_search — Tavily 웹 검색

In [ ]:
from backend.app.agent.tools import web_search

result = await web_search.ainvoke({
    "query": "제조업 AI 예지보전 시장 동향 2026",
    "max_results": 3,
})
search_results = json.loads(result)
if isinstance(search_results, list):
    print(f"검색 결과 수: {len(search_results)}")
    for r in search_results:
        print(f"  [{r['title']}] {r['url']}")
        print(f"    {r['content'][:100]}...")
else:
    print(f"에러: {search_results}")

#### 4-2. fetch_notion_deal — Notion 딜 정보 조회

실제 Notion page ID가 필요합니다. 테스트용 page ID를 설정하세요.

In [ ]:
# from backend.app.agent.tools import fetch_notion_deal
#
# # 실제 Notion page ID로 교체하세요
# TEST_PAGE_ID = "your-notion-page-id-here"
# result = await fetch_notion_deal.ainvoke({"page_id": TEST_PAGE_ID})
# print(result[:500])